# LSNM2024 Packet-Level Models (No Flow Reconstruction)

Trains a temporal XGBoost and a CNN+LSTM directly on **per-packet** LSNM2024
data - no flow reconstruction, no CICFlowMeter-style aggregation, no
dependency on this project's live `Flow`/`FlowManager` serving code.

**Why:** `fetch_lsnm2024_25feature.py` reconstructs flow-level features from
LSNM2024's raw packet CSVs (needed for the family models, which require the
same 25-feature schema as the deployed model), but that required a series of
approximations to work around real data-quality issues - unreliable TCP port
fields (~50% placeholder values), spoofed-source floods where standard
5-tuple grouping produces almost nothing, near-zero-duration bursts that
needed special handling. This notebook sidesteps all of that: instead of
building "flows" at all, each packet is its own data point (with a handful
of raw + short-rolling-history features), and the LSTM consumes a sequence
of consecutive raw packets directly, not a sequence of flows.

**Scope:** LSNM2024 only - not mixed with CIC-IDS2018/CIC-DDoS2019. Outputs
to `models/lsnm2024_packet_only/`, not wired into the live app.

Requires `data/lsnm2024/{syn_flood,icmp_flood,ddos_icmp,ddos_udp,ddos_raw,
normal_data}.csv` (extracted from the LSNM2024 "Dataset-Ready" zip - see
`fetch_lsnm2024_25feature.py`'s docstring for provenance/licensing).

In [1]:
import json
import pickle
import re
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
from xgboost import XGBClassifier

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
LSNM_DIR = PROJECT_ROOT / "data" / "lsnm2024"
OUT_DIR = PROJECT_ROOT / "models" / "lsnm2024_packet_only"

ATTACK_FILES = {
    "syn_flood": "Syn",
    "icmp_flood": "ICMP-Flood",
    "ddos_icmp": "DDOS-ICMP",
    "ddos_udp": "DDOS-UDP",
    "ddos_raw": "DDOS-RAW",
}
BENIGN_FILE = "normal_data"

HISTORY_WINDOW = 10       # packets of rolling history for the temporal-XGB features
SEQUENCE_LENGTH = 10      # raw packets per LSTM input sequence
ROLLING_STRIDE = 5        # step between consecutive LSTM sequences (overlapping windows, matches this project's existing sequence-building convention)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"training device: {DEVICE}")
if DEVICE.type == "cpu":
    print("WARNING: no GPU detected - CNN+LSTM training will be slower.")

training device: cuda


## Load + clean per-packet data (reused from fetch_lsnm2024_25feature.py, verified there against the real files)

In [2]:
IP_RE = re.compile(r"^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$")

NEEDED_COLUMNS = [
    "Frame Time (Epoch)", "IP Source", "IP Destination", "Protocol", "IP Protocol",
    "frame length", "IP Length", "TCP Length", "UDP Length",
]


def load_and_clean(path):
    df = pd.read_csv(path, usecols=lambda c: c in NEEDED_COLUMNS, low_memory=False)
    df = df.dropna(subset=["IP Source", "IP Destination", "Frame Time (Epoch)"])
    df = df[df["IP Source"].astype(str).str.match(IP_RE) & df["IP Destination"].astype(str).str.match(IP_RE)]

    df["Frame Time (Epoch)"] = pd.to_numeric(df["Frame Time (Epoch)"], errors="coerce")
    df = df.dropna(subset=["Frame Time (Epoch)"])

    for col in ["frame length", "IP Length", "TCP Length", "UDP Length"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df.sort_values("Frame Time (Epoch)").reset_index(drop=True)


def per_packet_features(df):
    """5 raw-ish features per packet - no flow aggregation. payload/header
    split verified directly against the real data: TCP Length is already
    pure payload bytes (IP Length - TCP Length = 40 = 20-byte IP header +
    20-byte TCP header exactly, no options); UDP Length includes the
    8-byte UDP header per Wireshark convention; ICMP/other subtracts a
    fixed IP(20)+ICMP(8) header, matching the live-serving fix already
    applied to feature_extraction/flow.py this session."""
    is_tcp = df["Protocol"].eq("TCP") | df["IP Protocol"].eq("TCP")
    is_udp = df["Protocol"].eq("UDP") | df["IP Protocol"].eq("UDP")

    protocol_num = np.select([is_tcp, is_udp], [6, 17], default=0)
    payload = np.select(
        [is_tcp, is_udp],
        [df["TCP Length"].fillna(0), (df["UDP Length"].fillna(8) - 8).clip(lower=0)],
        default=(df["IP Length"].fillna(28) - 28).clip(lower=0),
    )

    ts = df["Frame Time (Epoch)"].values
    delta_t_us = np.diff(ts, prepend=ts[0]) * 1_000_000
    delta_t_us = np.clip(delta_t_us, 0, None)

    return pd.DataFrame({
        "ts": ts,
        "frame_length": df["frame length"].fillna(0).values,
        "ip_length": df["IP Length"].fillna(0).values,
        "payload": payload,
        "protocol_num": protocol_num.astype(float),
        "delta_t_us": delta_t_us,
    })

## Build the combined, labeled packet dataset

Each attack file's own benign contribution comes from a non-overlapping
slice of the shared `normal_data.csv` pool (LSNM2024 provides ONE shared
benign file across all 15 attack types, not per-attack-type embedded
benign) - same convention as `fetch_lsnm2024_25feature.py`, so no single
benign packet is duplicated across two different attack types' training
data.

In [3]:
print("loading benign pool ...")
benign_pkts = per_packet_features(load_and_clean(LSNM_DIR / f"{BENIGN_FILE}.csv"))
benign_pkts["Label"] = "BENIGN"
print(f"  {len(benign_pkts)} benign packets")

n_chunks = len(ATTACK_FILES)
shuffled_benign = benign_pkts.sample(frac=1, random_state=42).reset_index(drop=True)
chunk_bounds = np.linspace(0, len(shuffled_benign), n_chunks + 1, dtype=int)

frames = []
for i, (stem, label) in enumerate(ATTACK_FILES.items()):
    print(f"loading {stem} ...")
    path = LSNM_DIR / f"{stem}.csv"
    pkts = per_packet_features(load_and_clean(path))
    pkts["Label"] = label
    print(f"  {len(pkts)} attack packets")

    benign_chunk = shuffled_benign.iloc[chunk_bounds[i]:chunk_bounds[i + 1]].copy()

    day_df = pd.concat([pkts, benign_chunk], ignore_index=True).sort_values("ts").reset_index(drop=True)
    day_df["day"] = stem
    frames.append(day_df)

df = pd.concat(frames, ignore_index=True)
df["Binary_Label"] = (df["Label"] != "BENIGN").astype(int)
print(f"\ncombined: {len(df)} packets across {len(frames)} day-tagged files")
print(df.groupby("day")["Binary_Label"].agg(["count", "mean"]))

loading benign pool ...
  1258422 benign packets
loading syn_flood ...
  199938 attack packets
loading icmp_flood ...
  240078 attack packets
loading ddos_icmp ...
  199977 attack packets
loading ddos_udp ...
  166325 attack packets
loading ddos_raw ...
  326478 attack packets

combined: 2391218 packets across 5 day-tagged files
             count      mean
day                         
ddos_icmp   451662  0.442758
ddos_raw    578163  0.564682
ddos_udp    418009  0.397898
icmp_flood  491762  0.488200
syn_flood   451622  0.442711


## Per-day chronological train/test split

Same convention as this project's other training scripts: split each
day/file on its own attack-timestamp quantile, not a single global cutoff -
avoids one file's train/test boundary leaking into another's.

In [4]:
def per_day_split(day_df):
    attacks = day_df.loc[day_df["Binary_Label"] == 1, "ts"]
    if len(attacks) < 20:
        cutoff = day_df["ts"].quantile(0.8)
    else:
        cutoff = np.quantile(attacks, 0.7)
    return (day_df["ts"] < cutoff).values


train_mask = np.zeros(len(df), dtype=bool)
pos = 0
for day, group in df.groupby("day", sort=False):
    mask = per_day_split(group)
    train_mask[pos:pos + len(group)] = mask
    pos += len(group)

test_mask = ~train_mask
print(f"train={train_mask.sum()}  test={test_mask.sum()}")
print(f"train attack ratio: {df.loc[train_mask, 'Binary_Label'].mean():.3f}")
print(f"test attack ratio:  {df.loc[test_mask, 'Binary_Label'].mean():.3f}")

train=1615030  test=776188
train attack ratio: 0.491
test attack ratio:  0.438


## Temporal XGBoost: own packet features + rolling history

Rolling stats over the PRECEDING `HISTORY_WINDOW` packets within the same
day/file - `shift(1)` first so the current packet never leaks into its own
history, mirroring `train_experimental_models_2019.ipynb`'s Variant 2
exactly, just computed per-packet instead of per-flow.

In [5]:
RAW_FEATURES = ["frame_length", "payload", "protocol_num", "delta_t_us"]

def build_rolling_features(df):
    df = df.sort_values(["day", "ts"]).reset_index(drop=True)
    grp = df.groupby("day", sort=False)

    hist_cols = []
    for col in ["frame_length", "payload", "delta_t_us"]:
        shifted = grp[col].shift(1)
        df["_shifted"] = shifted
        roll = df.groupby("day")["_shifted"]
        mean_col, std_col = f"hist_mean_{col}", f"hist_std_{col}"
        df[mean_col] = roll.rolling(window=HISTORY_WINDOW, min_periods=1).mean().reset_index(level=0, drop=True)
        df[std_col] = roll.rolling(window=HISTORY_WINDOW, min_periods=1).std().reset_index(level=0, drop=True)
        hist_cols += [mean_col, std_col]
    df.drop(columns=["_shifted"], inplace=True)

    df["hist_pkt_count"] = grp.cumcount().clip(upper=HISTORY_WINDOW)
    hist_cols.append("hist_pkt_count")

    df[hist_cols] = df[hist_cols].fillna(0)
    return df, hist_cols


df, hist_cols = build_rolling_features(df)
temporal_features = RAW_FEATURES + hist_cols
print(f"temporal XGB features ({len(temporal_features)}): {temporal_features}")

X = df[temporal_features]
y = df["Binary_Label"].values

xgb_temporal = XGBClassifier(n_estimators=200, max_depth=5, eval_metric="logloss", random_state=42)
xgb_temporal.fit(X[train_mask], y[train_mask])
probs_xgb = xgb_temporal.predict_proba(X[test_mask])[:, 1]

temporal XGB features (11): ['frame_length', 'payload', 'protocol_num', 'delta_t_us', 'hist_mean_frame_length', 'hist_std_frame_length', 'hist_mean_payload', 'hist_std_payload', 'hist_mean_delta_t_us', 'hist_std_delta_t_us', 'hist_pkt_count']


## Threshold tuning + evaluation (temporal XGBoost)

In [6]:
def tune_threshold(y_true, probs):
    return float(max(np.linspace(0.05, 0.95, 91), key=lambda v: f1_score(y_true, probs >= v, zero_division=0)))


def eval_at_threshold(y_true, probs, threshold):
    preds = (probs >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(y_true, preds),
        "precision": precision_score(y_true, preds, zero_division=0),
        "recall": recall_score(y_true, preds, zero_division=0),
        "f1": f1_score(y_true, preds, zero_division=0),
        "threshold": threshold,
    }


def feature_importance_report(names, importances):
    report = sorted(zip(names, [float(i) for i in importances]), key=lambda x: -x[1])
    shortcut_warning = report[0][1] > 0.5
    return report, shortcut_warning


y_test = y[test_mask]
thr_xgb = tune_threshold(y_test, probs_xgb)
metrics_xgb = eval_at_threshold(y_test, probs_xgb, thr_xgb)
imp_xgb, shortcut_xgb = feature_importance_report(temporal_features, xgb_temporal.feature_importances_)

print("temporal XGBoost metrics:", metrics_xgb)
print("feature importances:", imp_xgb)
print("shortcut_warning:", shortcut_xgb)

temporal XGBoost metrics: {'accuracy': 0.9988443521414915, 'precision': 0.9986242043001833, 'recall': 0.9999717886910267, 'f1': 0.9992975421807517, 'threshold': 0.08}
feature importances: [('hist_std_delta_t_us', 0.9561876654624939), ('hist_mean_delta_t_us', 0.043125223368406296), ('protocol_num', 0.00030860997503623366), ('hist_mean_frame_length', 0.0002520981361158192), ('hist_std_frame_length', 3.688764991238713e-05), ('delta_t_us', 3.0689770937897265e-05), ('frame_length', 1.8754304619506e-05), ('hist_std_payload', 1.3536543519876432e-05), ('hist_pkt_count', 1.313878055952955e-05), ('hist_mean_payload', 7.98891323938733e-06), ('payload', 5.425359631772153e-06)]


## CNN+LSTM: raw sequence of consecutive PACKETS (not flows)

Each input sequence is `SEQUENCE_LENGTH` consecutive raw packets (own
features only, no rolling history - the LSTM learns temporal structure
itself), built with overlapping stride `ROLLING_STRIDE` within each
day/file so the same packet can anchor multiple training sequences without
crossing a day/file boundary.

In [7]:
def build_packet_sequences(df, test_mask):
    """A sequence can straddle the train/test boundary within a day - each
    sequence is assigned to train/test by its OWN last packet's split
    membership (the packet whose label the sequence is predicting), not a
    per-day binary split."""
    df = df.sort_values(["day", "ts"]).reset_index(drop=True)
    test_mask_sorted = test_mask[df.index.values] if hasattr(test_mask, "__getitem__") else test_mask

    seqs, labels, is_test = [], [], []
    pos = 0
    for _, group in df.groupby("day", sort=False):
        n = len(group)
        vals = group[RAW_FEATURES].values.astype(np.float32)
        lbls = group["Binary_Label"].values
        group_test_mask = test_mask_sorted[pos:pos + n]
        pos += n

        for start in range(0, max(n - SEQUENCE_LENGTH + 1, 0), ROLLING_STRIDE):
            end = start + SEQUENCE_LENGTH
            seqs.append(vals[start:end])
            labels.append(lbls[end - 1])
            is_test.append(bool(group_test_mask[end - 1]))

    if not seqs:
        empty = np.zeros((0, SEQUENCE_LENGTH, len(RAW_FEATURES)), dtype=np.float32)
        return empty, np.array([]), np.array([], dtype=bool)
    return np.stack(seqs), np.array(labels), np.array(is_test, dtype=bool)


# df was already re-sorted by ["day","ts"] when building rolling features,
# and train_mask/test_mask were built over that same row order - reuse
# them directly rather than re-deriving anything.
X_seq, y_seq, seq_test_mask = build_packet_sequences(df, test_mask)
seq_train_mask = ~seq_test_mask
print(f"built {len(X_seq)} packet sequences of length {SEQUENCE_LENGTH} "
      f"(train={seq_train_mask.sum()} test={seq_test_mask.sum()})")

built 478236 packet sequences of length 10 (train=322998 test=155238)


In [8]:
scaler3 = StandardScaler().fit(X_seq[seq_train_mask].reshape(-1, len(RAW_FEATURES)))
X_seq_scaled = scaler3.transform(X_seq.reshape(-1, len(RAW_FEATURES))).reshape(X_seq.shape)


class CNN_LSTM(nn.Module):
    def __init__(self, n_features, seq_len=SEQUENCE_LENGTH):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels=n_features, out_channels=32, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.lstm = nn.LSTM(input_size=32, hidden_size=64, batch_first=True)
        self.fc = nn.Linear(64, 1)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.relu(self.conv1(x))
        x = x.permute(0, 2, 1)
        _, (h, _) = self.lstm(x)
        return self.fc(h.squeeze(0))


def predict_in_batches(model, X, device=DEVICE, batch_size=2048):
    model.eval()
    logits = []
    with torch.no_grad():
        for start in range(0, len(X), batch_size):
            xb = torch.FloatTensor(X[start:start + batch_size]).to(device)
            logits.append(model(xb).cpu())
    return torch.cat(logits, dim=0).numpy().ravel() if logits else np.array([])


def train_cnn_lstm(X_train, y_train, device=DEVICE):
    idx = np.arange(len(X_train))
    rng = np.random.default_rng(0)
    rng.shuffle(idx)
    split = int(len(idx) * 0.8)
    tr_idx, val_idx = idx[:split], idx[split:]

    train_loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_train[tr_idx]), torch.FloatTensor(y_train[tr_idx]).unsqueeze(1)),
        batch_size=512, shuffle=True,
    )
    val_loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_train[val_idx]), torch.FloatTensor(y_train[val_idx]).unsqueeze(1)),
        batch_size=512, shuffle=False,
    )

    model = CNN_LSTM(n_features=X_train.shape[2]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    pos_n = y_train[tr_idx].sum()
    neg_n = len(tr_idx) - pos_n
    pos_weight = torch.tensor([neg_n / max(pos_n, 1)]).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    best_val_loss, patience, counter, best_state = float("inf"), 3, 0, None
    for epoch in range(1, 21):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_losses = [criterion(model(xb.to(device)), yb.to(device)).item() for xb, yb in val_loader]
            val_loss = float(np.mean(val_losses)) if val_losses else float("inf")
        print(f"    epoch {epoch}: val_loss={val_loss:.4f}")
        if val_loss < best_val_loss:
            best_val_loss, best_state, counter = val_loss, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            counter += 1
            if counter >= patience:
                print("    early stopping")
                break
    model.load_state_dict(best_state)
    return model


print(f"training CNN+LSTM on {DEVICE} ...")
model3 = train_cnn_lstm(X_seq_scaled[seq_train_mask], y_seq[seq_train_mask])
logits3 = predict_in_batches(model3, X_seq_scaled[seq_test_mask])
probs_lstm = 1 / (1 + np.exp(-logits3))
y_seq_test = y_seq[seq_test_mask]
thr_lstm = tune_threshold(y_seq_test, probs_lstm)
metrics_lstm = eval_at_threshold(y_seq_test, probs_lstm, thr_lstm)
print("packet-sequence CNN+LSTM metrics:", metrics_lstm)

training CNN+LSTM on cuda ...
    epoch 1: val_loss=0.1438
    epoch 2: val_loss=0.1331
    epoch 3: val_loss=0.1301
    epoch 4: val_loss=0.1124
    epoch 5: val_loss=0.1035
    epoch 6: val_loss=0.1211
    epoch 7: val_loss=0.0959
    epoch 8: val_loss=0.0866
    epoch 9: val_loss=0.1274
    epoch 10: val_loss=0.0695
    epoch 11: val_loss=0.0772
    epoch 12: val_loss=0.1362
    epoch 13: val_loss=0.0481
    epoch 14: val_loss=0.0415
    epoch 15: val_loss=0.0348
    epoch 16: val_loss=0.0581
    epoch 17: val_loss=0.0383
    epoch 18: val_loss=0.0711
    early stopping
packet-sequence CNN+LSTM metrics: {'accuracy': 0.9989242324688543, 'precision': 0.999474818734078, 'recall': 0.9992163500720957, 'f1': 0.9993455676906376, 'threshold': 0.58}


## Summary

In [9]:
print(f"{'Model':<30}{'Threshold':<11}{'Accuracy':<10}{'Precision':<10}{'Recall':<10}{'F1':<10}")
for name, m in [("Temporal XGBoost (packets)", metrics_xgb), ("CNN+LSTM (packet sequence)", metrics_lstm)]:
    print(f"{name:<30}{m['threshold']:<11.3f}{m['accuracy']:<10.4f}{m['precision']:<10.4f}{m['recall']:<10.4f}{m['f1']:<10.4f}")

Model                         Threshold  Accuracy  Precision Recall    F1        
Temporal XGBoost (packets)    0.080      0.9988    0.9986    1.0000    0.9993    
CNN+LSTM (packet sequence)    0.580      0.9989    0.9995    0.9992    0.9993    


## Save artifacts + provenance

In [10]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

with open(OUT_DIR / "xgb_temporal.pkl", "wb") as f:
    pickle.dump(xgb_temporal, f)
with open(OUT_DIR / "threshold_xgb_temporal.pkl", "wb") as f:
    pickle.dump(thr_xgb, f)
with open(OUT_DIR / "features_xgb_temporal.pkl", "wb") as f:
    pickle.dump(temporal_features, f)

torch.save(model3.to("cpu").state_dict(), OUT_DIR / "cnn_lstm_packet.pt")
with open(OUT_DIR / "threshold_cnn_lstm.pkl", "wb") as f:
    pickle.dump(thr_lstm, f)
with open(OUT_DIR / "scaler_cnn_lstm.pkl", "wb") as f:
    pickle.dump(scaler3, f)
with open(OUT_DIR / "raw_features.pkl", "wb") as f:
    pickle.dump(RAW_FEATURES, f)

provenance = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "dataset": "LSNM2024 only, packet-level (no flow reconstruction) - Syn/ICMP-Flood/DDOS-ICMP/DDOS-UDP/DDOS-RAW + own-benign-chunk per file",
    "days": sorted(df["day"].unique().tolist()),
    "total_packets": int(len(df)),
    "train_packets": int(train_mask.sum()),
    "test_packets": int(test_mask.sum()),
    "temporal_xgb": {"metrics": metrics_xgb, "feature_importances": imp_xgb, "shortcut_warning": shortcut_xgb},
    "cnn_lstm_packet": {"metrics": metrics_lstm, "sequence_length": SEQUENCE_LENGTH, "stride": ROLLING_STRIDE,
                        "architecture": "Conv1d(4->32,k=3) + LSTM(32->64) + Linear(64->1)"},
    "note": ("Packet-level models - each timestep/row is one raw packet (frame_length, payload, "
             "protocol_num, delta_t_us [+ rolling history for the XGB]), not a reconstructed flow. "
             "No dependency on feature_extraction/flow.py or detection/experimental_history.py - "
             "not wired into the live serving path."),
}
with open(OUT_DIR / "provenance.json", "w", encoding="utf-8") as f:
    json.dump(provenance, f, indent=2, default=str)
print(f"saved artifacts + provenance to {OUT_DIR}/")

saved artifacts + provenance to c:\Users\udaya\OneDrive\Desktop\Major Project\review\Adaptive-IDS-with-XAI\models\lsnm2024_packet_only/
